### About Dataset

Context This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content 5-core dataset of product reviews from Amazon Kindle Store category from May 1996 -
July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each
product has at least 5 reviews in this dataset. Columns

1. asin - ID of the product, like B000FA64PK
2. helpful - helpfulness rating of the review - example: 2/3.
3. overall - rating of the product.
4. reviewText - text of the review (heading).
5. reviewTime - time of the review (raw).
6. reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
7. reviewerName - name of the reviewer.
8. summary - summary of the review (description).
9. unixReviewTime - unix timestamp.

Acknowledgements This dataset is taken from Amazon product data, Julian McAuley, UCSD
website. http://jmcauley.ucsd.edu/data/amazon/

- License to the data files belong to them.
 ##### Inspiration
 - Sentiment analysis on reviews.
 - Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.
 - Fake reviews/ outliers. -Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr). 
 - Any other interesting analysis


### Best practicse
1. preprocessing and cleaning
2. Train test split
3. BOW,TF-IDF,WORD2Vec
4. Train ML algorithm

In [1]:
import pandas as pd
data =pd.read_csv('all_kindle_review.csv')

In [2]:
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [3]:
#we required only two
data = data[['reviewText','rating']]
data.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [4]:
data.shape

(12000, 2)

In [5]:
data.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [6]:
data['rating'].unique()

array([3, 5, 4, 2, 1])

In [7]:
data['rating'].value_counts()


rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

## Preprocessing and Cleaning

In [8]:
## converting Positive review is 1 when rating less than 3 and negative review is 0 when greater than 3
data['rating']=data['rating'].apply(lambda x:0 if x<3 else 1)

In [9]:
data.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",1
1,Great short read. I didn't want to put it dow...,1
2,I'll start by saying this is the first of four...,1
3,Aggie is Angela Lansbury who carries pocketboo...,1
4,I did not expect this type of book to be in li...,1


In [10]:
data['rating'].unique()

array([1, 0])

In [11]:
## 1. lower all the casses
data['reviewText'] = data['reviewText'].str.lower()

In [12]:
data.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [13]:
import re
import nltk
from nltk.corpus import stopwords

In [14]:
from bs4 import BeautifulSoup

In [15]:
## 2.Removing special characters

data['reviewText']=data['reviewText'].apply(lambda x:re.sub('[^a-z A-z 0-9-]+', '',x))
## Remove the stopswords
data['reviewText']=data['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]))
## Remove url 
data['reviewText']=data['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
## Remove html tags
data['reviewText']=data['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())
## Remove any additional spaces
data['reviewText']=data['reviewText'].apply(lambda x: " ".join(x.split()))

In [16]:
data.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [17]:
## lemmatizer
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [18]:
def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [19]:
data['reviewText']=data['reviewText'].apply(lambda x:lemmatize_words(x))

In [20]:
data.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


## Train test split

In [22]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(data['reviewText'],data['rating'],test_size=.20,random_state=30)

In [26]:
from sklearn.feature_extraction.text import CountVectorizer
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train).toarray()
X_test_bow = bow.transform(X_test).toarray()

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

In [28]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(9600, 35931))

In [29]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow = GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf = GaussianNB().fit(X_train_tfidf,y_train)

In [30]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

In [31]:
y_pred_bow = nb_model_bow.predict(X_test_bow)

In [32]:
y_pred_tfidf = nb_model_tfidf.predict(X_test_tfidf)

In [34]:
print(f"accuracy_score BOW:",accuracy_score(y_test,y_pred_bow))
print(f"confusion_matrix BOW:",confusion_matrix(y_test,y_pred_bow))
print(f"classification_report BOW:",classification_report(y_test,y_pred_bow))

accuracy_score BOW: 0.5795833333333333
confusion_matrix BOW: [[530 283]
 [726 861]]
classification_report BOW:               precision    recall  f1-score   support

           0       0.42      0.65      0.51       813
           1       0.75      0.54      0.63      1587

    accuracy                           0.58      2400
   macro avg       0.59      0.60      0.57      2400
weighted avg       0.64      0.58      0.59      2400



In [36]:
print(f"accuracy_score TFIDF:",accuracy_score(y_test,y_pred_tfidf))
print(f"confusion_matrix TFIDF:",confusion_matrix(y_test,y_pred_tfidf))
print(f"classification_report TFIDF:",classification_report(y_test,y_pred_tfidf))

accuracy_score TFIDF: 0.5833333333333334
confusion_matrix TFIDF: [[522 291]
 [709 878]]
classification_report TFIDF:               precision    recall  f1-score   support

           0       0.42      0.64      0.51       813
           1       0.75      0.55      0.64      1587

    accuracy                           0.58      2400
   macro avg       0.59      0.60      0.57      2400
weighted avg       0.64      0.58      0.59      2400



### AVERAGE WORD 2 VEC

In [39]:
import gensim
model = gensim.models.Word2Vec(data['reviewText'])

In [40]:
def avg_word2vec(doc):
    return np.mean([model.wv[word] for word in doc if word in model.wv.index_to_key],axis=0)

In [41]:
from tqdm import tqdm

In [43]:
import numpy as np
X = []
for i in tqdm(range(len(data['reviewText']))):
    X.append(avg_word2vec(data['reviewText'][i]))

100%|██████████| 12000/12000 [00:04<00:00, 2912.94it/s]


In [54]:
#independent feature
#X

In [46]:
data['rating'].shape

(12000,)

In [47]:
#dependent feature
y = pd.get_dummies(data['rating'])
y = y.iloc[:,0].values

In [48]:
y.shape

(12000,)

In [49]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.20,random_state=40)

In [51]:
from sklearn.naive_bayes import GaussianNB
nb_model_new = GaussianNB()
nb_model_new.fit(X_train,y_train)

GaussianNB()

In [52]:
y_pred_new = nb_model_new.predict(X_test)

In [53]:
print(f"accuracy_score :",accuracy_score(y_test,y_pred_new))
print(f"confusion_matrix :",confusion_matrix(y_test,y_pred_new))
print(f"classification_report :",classification_report(y_test,y_pred_new))

accuracy_score : 0.5929166666666666
confusion_matrix : [[1007  582]
 [ 395  416]]
classification_report :               precision    recall  f1-score   support

       False       0.72      0.63      0.67      1589
        True       0.42      0.51      0.46       811

    accuracy                           0.59      2400
   macro avg       0.57      0.57      0.57      2400
weighted avg       0.62      0.59      0.60      2400

